In [1]:
import polars as pl
import pandas as pd
import numpy as np
import itertools
from skbio.diversity import alpha_diversity 
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [3]:
metadata_sh = pd.read_csv('data/metadata_urbansoil.csv', index_col='Sample')
metadata_sh.loc[metadata_sh['Site'].str.startswith('NanJing'), 'Site'] = 'Nanjing'
metadata_sh.loc[metadata_sh['Site'].str.startswith('ShangHai'), 'Site'] = 'Shanghai'
metadata_sh["Ecosystem"] = "Urban"
metadata_sh

,Site,Ecosystem
Sample,,
CPSNJ01_350,Nanjing,Urban
CPSNJ02_350,Nanjing,Urban
CPSNJ03_350,Nanjing,Urban
CPSNJ04_350,Nanjing,Urban
CPSNJ08_350,Nanjing,Urban
CPSNJ09_350,Nanjing,Urban
CPSNJ10_350,Nanjing,Urban
CPSNJ11_350,Nanjing,Urban
CPSNJ13_350,Nanjing,Urban


In [4]:
metadata_dog= pd.read_csv('data/metadata_dog.csv',index_col=0)
metadata_dog.index.name = "Sample"
metadata_dog["Ecosystem"] = "Canine Gut"
metadata_dog

,Site,Ecosystem
Sample,,
D000_350,Dog,Canine Gut
D001_350,Dog,Canine Gut
D002_350,Dog,Canine Gut
D003_350,Dog,Canine Gut
D004_350,Dog,Canine Gut
D005_350,Dog,Canine Gut
D006_350,Dog,Canine Gut
D007_350,Dog,Canine Gut
D008_350,Dog,Canine Gut


In [5]:
ena_smag = pd.read_csv('data/ena_meta_smag.tsv', sep='\t')
ena_smag

,run_accession,study_accession,sample_accession,experiment_accession,tax_id,scientific_name,fastq_ftp,submitted_ftp,sra_ftp,bam_ftp
0,SRR25158175,PRJNA983538,SAMN35729566,SRX20906449,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/075/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/075/SRR25158...,NaN
1,SRR25158178,PRJNA983538,SAMN35729644,SRX20906446,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/078/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/078/SRR25158...,NaN
2,SRR25158180,PRJNA983538,SAMN35729642,SRX20906444,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/080/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/080/SRR25158...,NaN
3,SRR25158183,PRJNA983538,SAMN35729639,SRX20906441,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/083/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/083/SRR25158...,NaN
4,SRR25158186,PRJNA983538,SAMN35729565,SRX20906438,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/086/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/086/SRR25158...,NaN
...,...,...,...,...,...,...,...,...,...,...
358,SRR25158523,PRJNA983538,SAMN35729667,SRX20906464,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/023/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/023/SRR25158...,NaN
359,SRR25158526,PRJNA983538,SAMN35729665,SRX20906461,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/026/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/026/SRR25158...,NaN
360,SRR25158531,PRJNA983538,SAMN35729660,SRX20906456,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/031/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/031/SRR25158...,NaN
361,SRR25158534,PRJNA983538,SAMN35729657,SRX20906453,410658,soil metagenome,ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/034/SRR251...,NaN,ftp.sra.ebi.ac.uk/vol1/srr/SRR251/034/SRR25158...,NaN


In [6]:
metadata_smag = pd.read_csv('data/metadata_smag.csv', skiprows=[0])
metadata_smag = metadata_smag.drop(columns=[col for col in metadata_smag.columns if col not in ["Ecosystem", "Biosample"]])
metadata_smag = metadata_smag.merge(ena_smag, left_on="Biosample", right_on="sample_accession")
metadata_smag = metadata_smag.drop(columns=[col for col in metadata_smag.columns if col not in ["Ecosystem", "run_accession"]]).rename(columns={"run_accession":"Sample"}).set_index("Sample")
metadata_smag["Site"] = "SMAG"
metadata_smag

,Ecosystem,Site
Sample,,
SRR25158537,Forest,SMAG
SRR25158536,Forest,SMAG
SRR25158457,Grassland,SMAG
SRR25158273,Agricultural Land,SMAG
SRR25158433,Wetland,SMAG
...,...,...
SRR25158336,Wetland,SMAG
SRR25158441,Wetland,SMAG
SRR25158439,Grassland,SMAG


In [7]:
metadata = pd.concat([metadata_sh, metadata_smag, metadata_dog])
metadata

,Site,Ecosystem
Sample,,
CPSNJ01_350,Nanjing,Urban
CPSNJ02_350,Nanjing,Urban
CPSNJ03_350,Nanjing,Urban
CPSNJ04_350,Nanjing,Urban
CPSNJ08_350,Nanjing,Urban
...,...,...
D048_350,Dog,Canine Gut
D049_350,Dog,Canine Gut
D050_350,Dog,Canine Gut


In [8]:
alpha = pd.read_csv('alpha_diversity/alpha_diversity.tsv', sep='\t')
alpha = alpha.merge(metadata, left_on="sample", right_on="Sample", how="left")
alpha = alpha.set_index('sample').rename(columns={"Chao1":"chao1", "Shannon":"shannon","Simpson":"simpson"})

In [9]:
alpha.to_csv('../../../analysis/pre-calculated_data/alpha_diversity.csv')
alpha

,chao1,simpson,shannon,Site,Ecosystem
sample,,,,,
SRR25158350,2815.955027,0.971360,6.893876,SMAG,Wetland
SRR25158304,2518.101709,0.971311,6.556217,SMAG,Bare Land
SRR25158536,3913.855805,0.971324,7.321940,SMAG,Forest
SRR25158444,4700.153402,0.971383,7.310066,SMAG,Agricultural Land
SRR25158393,4808.474763,0.971378,7.424590,SMAG,Agricultural Land
...,...,...,...,...,...
D044_350,1419.713475,0.971202,3.251397,Dog,Canine Gut
D032_350,753.296915,0.971105,2.611935,Dog,Canine Gut
D036_350,1097.076559,0.971146,3.510075,Dog,Canine Gut
